# detinfer GPU Verification

This notebook tests detinfer on a real GPU to verify it enforces determinism.

**Before running:** Go to Runtime > Change runtime type > Select **T4 GPU**

---

## Step 1: Check GPU

In [ ]:
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')
    print(f'PyTorch version: {torch.__version__}')
else:
    print('NO GPU detected.')
    print('Go to Runtime > Change runtime type > T4 GPU')

## Step 2: Install detinfer

In [ ]:
!git clone -b v2-enforcement https://github.com/xailong-6969/detinfer.git
%cd detinfer
!pip install -e ".[transformers]" -q
print('detinfer installed!')

## Step 3: Choose a model

Change the model name below to any HuggingFace model you want to test.

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'  # Change this to any model

## Step 4: Test WITHOUT detinfer

This shows what happens without detinfer — run the same prompt 5 times and compare hashes.

In [ ]:
import hashlib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
model.to('cuda')
model.eval()

prompt = 'What is 2 + 2? Answer with just the number.'

print('\n--- WITHOUT detinfer ---')
hashes = []
for i in range(5):
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    output_bytes = output.cpu().numpy().tobytes()
    h = hashlib.sha256(output_bytes).hexdigest()
    hashes.append(h)
    match = '' if i == 0 else (' (same)' if h == hashes[0] else ' (DIFFERENT!)')
    print(f'  Run {i+1}: {h[:32]}...{match}')

unique = len(set(hashes))
if unique == 1:
    print(f'\nResult: All {len(hashes)} hashes match on this GPU')
    print('(Greedy decode may be stable, but internal float values can still differ across GPUs)')
else:
    print(f'\nResult: NON-DETERMINISTIC! {unique} different hashes')

del model, tokenizer
torch.cuda.empty_cache()

## Step 5: Test WITH detinfer

Load the same model through detinfer and verify determinism.

In [ ]:
from detinfer import DeterministicEngine

engine = DeterministicEngine(seed=42, precision='high')
report = engine.load(MODEL_NAME)

print('--- Enforcement Report ---')
print(report)
print(f'\nDevice: {engine.device}')
print(f'Engine: {engine}')

In [ ]:
# Verify determinism — 5 runs
print('\n--- WITH detinfer ---')
result = engine.verify(prompt='What is 2 + 2? Answer with just the number.', num_runs=5)
print(result)

## Step 6: Test with multiple prompts

In [ ]:
test_prompts = [
    'What is 2 + 2? Answer with just the number.',
    'Write hello world in Python.',
    'What is the capital of France? One word.',
    'Explain gravity in one sentence.',
    'Translate hello to Spanish.',
    'What is 15 * 7?',
    'Write a function that adds two numbers in Python.',
    'Is the sky blue? Yes or no.',
]

print('--- Multi-Prompt Verification ---\n')
all_deterministic = True

for i, prompt in enumerate(test_prompts):
    result = engine.verify(prompt=prompt, num_runs=3)
    status = 'PASS' if result.is_deterministic else 'FAIL'
    print(f'  [{status}] Prompt {i+1}: "{prompt[:50]}"')
    if not result.is_deterministic:
        all_deterministic = False

print(f'\n{"ALL DETERMINISTIC" if all_deterministic else "SOME FAILED"} across {len(test_prompts)} prompts')

## Step 7: Show environment info

In [ ]:
from detinfer.guardian import EnvironmentGuardian

guardian = EnvironmentGuardian()
fingerprint = guardian.create_fingerprint()
print(fingerprint)

## Step 8: Export proof

Download this proof.json and verify it on another machine with `detinfer cross-verify proof.json`

In [ ]:
from detinfer.proof import create_proof

proof = create_proof(engine, 'What is 2 + 2? Answer with just the number.')
proof.save('proof.json')

print(proof)
print('\nproof.json saved! Download it and run on another machine:')
print('  detinfer cross-verify proof.json')